# GroupDNA — Hostel Friends Chat Analyzer
** Final Report is in the last section/feature of this notebook ⬇⬇⬇

In [1]:
import numpy as np

file_path = "/content/hostel_friends_chat.txt"

print("File path:", file_path)


File path: /content/hostel_friends_chat.txt


## Feature 1 - Chat Parser

In [2]:
messages = []
system_messages = 0
media_messages = 0
deleted_messages = 0

with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

for line in lines:
    line = line.strip()

    if line == "":
        continue

    if " - " not in line:
        system_messages += 1
        continue

    timestamp_text, rest = line.split(" - ", 1)

    if ": " not in rest:
        system_messages += 1
        continue

    sender, text = rest.split(": ", 1)

    if "<Media omitted>" in text:
        media_messages += 1
        continue

    if "This message was deleted" in text:
        deleted_messages += 1
        continue

    messages.append({
        "timestamp": timestamp_text,
        "sender": sender,
        "text": text
    })

participants = []

for message in messages:
    if message["sender"] not in participants:
        participants.append(message["sender"])

print("Successfully parsed", len(messages), "messages from",
      len(participants), "participants.")
print("Skipped", system_messages, "system messages,",
      media_messages, "media-omitted,", deleted_messages, "deleted messages.")

print("\nFirst 5 parsed messages:")
for message in messages[:5]:
    print(message)

print("\nLast 5 parsed messages:")
for message in messages[-5:]:
    print(message)


Successfully parsed 324 messages from 6 participants.
Skipped 0 system messages, 14 media-omitted, 0 deleted messages.

First 5 parsed messages:
{'timestamp': '01/04/24, 08:10', 'sender': 'Rahul', 'text': 'Good morning guys'}
{'timestamp': '01/04/24, 08:14', 'sender': 'Priya', 'text': 'Good morning'}
{'timestamp': '01/04/24, 12:30', 'sender': 'Aman', 'text': 'Lunch?'}
{'timestamp': '01/04/24, 12:34', 'sender': 'Karan', 'text': 'Yes bro'}
{'timestamp': '01/04/24, 18:10', 'sender': 'Rahul', 'text': 'Cricket today?'}

Last 5 parsed messages:
{'timestamp': '10/05/24, 11:25', 'sender': 'Aman', 'text': 'Thanks'}
{'timestamp': '10/05/24, 16:40', 'sender': 'Neha', 'text': 'What time?'}
{'timestamp': '10/05/24, 16:45', 'sender': 'Priya', 'text': 'At 5'}
{'timestamp': '10/05/24, 20:15', 'sender': 'Karan', 'text': 'Meeting done'}
{'timestamp': '10/05/24, 23:45', 'sender': 'Aman', 'text': 'Still working lol'}


## Feature 2 - Group Overview

In [3]:
message_count = {}
word_count = {}
message_length = {}

for person in participants:
    message_count[person] = 0
    word_count[person] = 0
    message_length[person] = 0

for message in messages:
    person = message["sender"]
    words = message["text"].split()

    message_count[person] += 1
    word_count[person] += len(words)
    message_length[person] += len(message["text"])

first_date = messages[0]["timestamp"].split(",")[0]
last_date = messages[-1]["timestamp"].split(",")[0]

day_parts_1 = first_date.split("/")
day_parts_2 = last_date.split("/")

# Number of calendar days is used only for the report period.
# The chat dataset uses consecutive dates in this project.
total_days = len(set(message["timestamp"].split(",")[0] for message in messages))

ranked_people = sorted(
    message_count.items(),
    key=lambda item: item[1],
    reverse=True
)

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)
print("Period :", first_date, "to", last_date)
print("Total messages :", len(messages))
print("Participants :", len(participants))
print("Total days :", total_days)

print("\nMESSAGES PER PERSON")

for person, count in ranked_people:
    percentage = count / len(messages) * 100
    print(f"{person:<10}: {count:>3} ({percentage:>4.1f}%)")




GROUP OVERVIEW
Period : 01/04/24 to 10/05/24
Total messages : 324
Participants : 6
Total days : 40

MESSAGES PER PERSON
Rahul     :  70 (21.6%)
Priya     :  70 (21.6%)
Aman      :  64 (19.8%)
Neha      :  57 (17.6%)
Karan     :  51 (15.7%)
Vikas     :  12 ( 3.7%)


## Feature 3 - Most Active Day and Hour

In [4]:
day_count = {}
hour_count = {}

for message in messages:
    date = message["timestamp"].split(",")[0]
    hour = int(message["timestamp"].split(",")[1].split(":")[0])

    if date not in day_count:
        day_count[date] = 0
    day_count[date] += 1

    if hour not in hour_count:
        hour_count[hour] = 0
    hour_count[hour] += 1

busiest_day = max(day_count, key=day_count.get)
busiest_hour = max(hour_count, key=hour_count.get)

print("Busiest day :", busiest_day, "(", day_count[busiest_day], "messages)")
print(
    "Busiest hour :",
    f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00",
    "(",
    hour_count[busiest_hour],
    "messages)"
)


Busiest day : 01/04/24 ( 13 messages)
Busiest hour : 18:00 - 19:00 ( 40 messages)


## Feature 4 - Activity Heatmap (NumPy)

In [5]:
activity = np.zeros((len(participants), 24), dtype=int)

for message in messages:
    person = message["sender"]
    hour = int(message["timestamp"].split(",")[1].split(":")[0])
    row = participants.index(person)
    activity[row][hour] += 1

print("ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print(" " * 10 + " ".join(f"{hour:02d}" for hour in range(24)))

maximum = activity.max()

for i in range(len(participants)):
    row_output = ""

    for value in activity[i]:
        if value == 0:
            symbol = "."
        elif value <= maximum * 0.25:
            symbol = "░"
        elif value <= maximum * 0.50:
            symbol = "▒"
        else:
            symbol = "█"

        row_output += symbol + " "

    print(f"{participants[i]:<10} {row_output}")




ACTIVITY HEATMAP (hour of day, columns 00 to 23)
          00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul      . . . . . . . ▒ ▒ ▒ ▒ . . ▒ ▒ . . ░ █ . . . ▒ . 
Priya      . . . . . . . . ▒ ▒ █ ▒ █ . . . ▒ ░ . ▒ ▒ . . . 
Aman       . . . . . . . . . . ▒ ▒ ▒ ▒ . . . ░ ▒ ▒ ▒ . ▒ ▒ 
Karan      . . . . . . . . . . . . █ . ▒ ▒ . ░ █ . ▒ . . . 
Neha       . . . . . . . . . █ █ . . . . ▒ ▒ ░ . . . █ . . 
Vikas      . . . . . . . ▒ . . . . . . . . . . . . . . . ░ 


## Feature 5 - Top Words

In [6]:
stop_words = [
    "i", "is", "the", "a", "an", "and", "or", "to",
    "of", "in", "on", "for", "with", "this", "that"
]

word_frequency = {}

for message in messages:
    text = message["text"].lower()

    for symbol in ".,!?;:()[]{}\"'":
        text = text.replace(symbol, "")

    for word in text.split():
        if word not in stop_words and word != "":
            if word not in word_frequency:
                word_frequency[word] = 0
            word_frequency[word] += 1

top_words = sorted(
    word_frequency.items(),
    key=lambda item: item[1],
    reverse=True
)

print("THIS GROUP'S FAVOURITE WORDS")

for word, count in top_words[:10]:
    bar_length = int(count / top_words[0][1] * 20)
    print(f"{word:<12} {'█' * bar_length} {count}")





THIS GROUP'S FAVOURITE WORDS
good         ████████████████████ 46
yes          ██████████████ 33
morning      ███████████ 26
guys         █████████ 22
anyone       ████████ 20
today        ███████ 18
lunch        ██████ 14
done         ██████ 14
meeting      ██████ 14
everyone     █████ 13


## Feature 6 - Response Speed and Silent Streaks

In [7]:
from datetime import datetime, timedelta

for message in messages:
    message["time"] = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

response_times = {}

for person in participants:
    response_times[person] = []

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    if previous["sender"] != current["sender"]:
        gap = current["time"] - previous["time"]
        minutes = gap.total_seconds() / 60
        response_times[current["sender"]].append(minutes)

average_response = {}

for person in participants:
    if len(response_times[person]) > 0:
        average_response[person] = (
            sum(response_times[person]) / len(response_times[person])
        )
    else:
        average_response[person] = 0

active_days = {}

for person in participants:
    active_days[person] = set()

for message in messages:
    day = message["time"].date()
    active_days[message["sender"]].add(day)

first_day = messages[0]["time"].date()
last_day = messages[-1]["time"].date()
chat_days = (last_day - first_day).days + 1

silent_streak = {}

for person in participants:
    longest = 0
    current = 0

    for day_number in range(chat_days):
        day = first_day + timedelta(days=day_number)

        if day not in active_days[person]:
            current += 1

            if current > longest:
                longest = current
        else:
            current = 0

    silent_streak[person] = longest

fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)

print("RESPONSE PATTERNS")
print(
    f"Fastest replier : {fastest} "
    f"(avg {average_response[fastest]:.1f} minutes)"
)
print(
    f"Slowest replier : {slowest} "
    f"(avg {average_response[slowest] / 60:.1f} hours)"
)

print("\nLONGEST SILENT STREAKS")

ranked_silent = sorted(
    silent_streak.items(),
    key=lambda item: item[1],
    reverse=True
)

for person, days in ranked_silent:
    print(f"{person:<10}: {days} days")


RESPONSE PATTERNS
Fastest replier : Karan (avg 135.8 minutes)
Slowest replier : Vikas (avg 7.7 hours)

LONGEST SILENT STREAKS
Vikas     : 5 days
Rahul     : 0 days
Priya     : 0 days
Aman      : 0 days
Karan     : 0 days
Neha      : 0 days


## Feature 7 - Personality Archetype Detection

In [8]:
# FEATURE 7 — PERSONALITY ARCHETYPES

def spammer_score(person):
    bursts = []
    burst = 0

    for message in messages:
        if message["sender"] == person:
            burst += 1
        else:
            if burst > 0:
                bursts.append(burst)
            burst = 0

    if burst > 0:
        bursts.append(burst)

    if len(bursts) == 0:
        return 0

    return sum(bursts) / len(bursts)


def group_mom_score(person):
    caring_words = [
        "okay", "safe", "eat", "sleep", "take care",
        "are you", "please", "reminder",
        "drink water", "don't forget"
    ]

    score = 0

    for message in messages:
        if message["sender"] == person:
            text = message["text"].lower()

            for word in caring_words:
                if word in text:
                    score += 1
                    break

    return score


def night_owl_score(person):
    total = 0
    night = 0

    for message in messages:
        if message["sender"] == person:
            total += 1

            if message["time"].hour >= 23 or message["time"].hour <= 4:
                night += 1

    if total == 0:
        return 0

    return night / total * 100


def storyteller_score(person):
    total = 0
    words = 0

    for message in messages:
        if message["sender"] == person:
            total += 1
            words += len(message["text"].split())

    if total == 0:
        return 0

    return words / total


def drama_queen_score(person):
    total = 0
    drama = 0

    for message in messages:
        if message["sender"] == person:
            total += 1
            text = message["text"]

            letters = False

            for character in text:
                if character.isalpha():
                    letters = True
                    break

            if (len(text) >= 3 and letters and text.upper() == text) or text.count("!") >= 2:
                drama += 1

    if total == 0:
        return 0

    return drama / total * 100


def ghost_score(person):
    return silent_streak[person] / chat_days * 100


def comedian_score(person):
    funny_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

    total = 0
    funny = 0

    for message in messages:
        if message["sender"] == person:
            total += 1
            text = message["text"].lower()

            for word in funny_words:
                if word in text:
                    funny += 1
                    break

    if total == 0:
        return 0

    return funny / total * 100


def question_master_score(person):
    total = 0
    questions = 0

    for message in messages:
        if message["sender"] == person:
            total += 1

            if message["text"].strip().endswith("?"):
                questions += 1

    if total == 0:
        return 0

    return questions / total * 100


# Calculate values

scores = {}

for person in participants:
    scores[person] = {
        "THE SPAMMER": spammer_score(person),
        "THE GROUP MOM": group_mom_score(person),
        "THE NIGHT OWL": night_owl_score(person),
        "THE STORYTELLER": storyteller_score(person),
        "THE DRAMA QUEEN": drama_queen_score(person),
        "THE GHOST": ghost_score(person),
        "THE COMEDIAN": comedian_score(person),
        "THE QUESTION MASTER": question_master_score(person)
    }


# Assign one archetype to each person.
# The six main archetypes are checked first.
# Comedian and Question Master are used as fallback/tiebreaker.

archetypes = {}

max_caring = 1

for person in participants:
    if scores[person]["THE GROUP MOM"] > max_caring:
        max_caring = scores[person]["THE GROUP MOM"]


for person in participants:

    main_scores = {}

    # Spammer: average burst should be greater than 3
    if scores[person]["THE SPAMMER"] > 3:
        main_scores["THE SPAMMER"] = scores[person]["THE SPAMMER"] / 3

    # Group Mom: higher caring-keyword count is stronger
    if scores[person]["THE GROUP MOM"] > 0:
        main_scores["THE GROUP MOM"] = (
            scores[person]["THE GROUP MOM"] / max_caring
        )

    # Night Owl: more than 60% late-night messages
    if scores[person]["THE NIGHT OWL"] > 60:
        main_scores["THE NIGHT OWL"] = (
            scores[person]["THE NIGHT OWL"] / 60
        )

    # Storyteller: average words greater than 30
    if scores[person]["THE STORYTELLER"] > 30:
        main_scores["THE STORYTELLER"] = (
            scores[person]["THE STORYTELLER"] / 30
        )

    # Drama Queen: more than 30% dramatic messages
    if scores[person]["THE DRAMA QUEEN"] > 30:
        main_scores["THE DRAMA QUEEN"] = (
            scores[person]["THE DRAMA QUEEN"] / 30
        )

    # Ghost: silent on more than 60% of days
    if scores[person]["THE GHOST"] > 60:
        main_scores["THE GHOST"] = (
            scores[person]["THE GHOST"] / 60
        )

    if len(main_scores) > 0:
        archetypes[person] = max(
            main_scores,
            key=main_scores.get
        )
    else:
        # Fallback archetypes
        fallback_scores = {
            "THE COMEDIAN": scores[person]["THE COMEDIAN"],
            "THE QUESTION MASTER": scores[person]["THE QUESTION MASTER"]
        }

        archetypes[person] = max(
            fallback_scores,
            key=fallback_scores.get
        )


# Output

print("PERSONALITY ARCHETYPES")

for person in participants:

    archetype = archetypes[person]
    value = scores[person][archetype]

    if archetype == "THE SPAMMER":
        print(f"{person} → {archetype} (avg {value:.1f} msgs in a row)")

    elif archetype == "THE GROUP MOM":
        print(f"{person} → {archetype} ({value:.0f} caring keywords)")

    elif archetype == "THE NIGHT OWL":
        print(f"{person} → {archetype} ({value:.1f}% msgs after 11 PM)")

    elif archetype == "THE STORYTELLER":
        print(f"{person} → {archetype} (avg {value:.1f} words per msg)")

    elif archetype == "THE DRAMA QUEEN":
        print(f"{person} → {archetype} ({value:.1f}% ALL-CAPS msgs)")

    elif archetype == "THE GHOST":
        print(f"{person} → {archetype} (silent {value:.1f}% of days)")

    elif archetype == "THE COMEDIAN":
        print(f"{person} → {archetype} ({value:.1f}% funny messages)")

    elif archetype == "THE QUESTION MASTER":
        print(f"{person} → {archetype} ({value:.1f}% questions)")


PERSONALITY ARCHETYPES
Rahul → THE QUESTION MASTER (28.6% questions)
Priya → THE GROUP MOM (7 caring keywords)
Aman → THE QUESTION MASTER (31.2% questions)
Karan → THE GROUP MOM (6 caring keywords)
Neha → THE GROUP MOM (7 caring keywords)
Vikas → THE QUESTION MASTER (58.3% questions)


## Feature 8 - Final Report

In [9]:
print("=" * 60)
print('GROUPDNA REPORT — "Hostel Friends"')
print("=" * 60)

print(
    len(set(message["time"].date() for message in messages)),
    "days •",
    len(messages),
    "messages •",
    len(participants),
    "members"
)

print("Period :", first_day.strftime("%d %B %Y"),
      "to", last_day.strftime("%d %B %Y"))

print(
    "Busiest day :",
    busiest_day,
    "(",
    day_count[busiest_day],
    "messages)"
)

print(
    "Busiest hour :",
    f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00"
)

print("\nMESSAGES PER PERSON")

maximum_count = ranked_people[0][1]

for person, count in ranked_people:
    bar_length = int(count / maximum_count * 20)

    print(
        f"{person:<10} "
        f"{'█' * bar_length} "
        f"{count} ({count / len(messages) * 100:.1f}%)"
    )

print("\nACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print(" " * 10 + " ".join(f"{hour:02d}" for hour in range(24)))

for i in range(len(participants)):
    row_output = ""

    for value in activity[i]:
        if value == 0:
            symbol = "."
        elif value <= maximum * 0.25:
            symbol = "░"
        elif value <= maximum * 0.50:
            symbol = "▒"
        else:
            symbol = "█"

        row_output += symbol + " "

    print(f"{participants[i]:<10} {row_output}")

print("\nTHIS GROUP'S FAVOURITE WORDS")

for word, count in top_words[:5]:
    bar_length = int(count / top_words[0][1] * 20)
    print(f"{word:<12} {'█' * bar_length} {count}")

print("\nRESPONSE PATTERNS")
print(
    f"Fastest replier : {fastest} "
    f"(avg {average_response[fastest]:.1f} minutes)"
)
print(
    f"Slowest replier : {slowest} "
    f"(avg {average_response[slowest] / 60:.1f} hours)"
)

print("\nLONGEST SILENT STREAKS")

for person, days in ranked_silent:
    print(f"{person:<10}: {days} days")

print("\nPERSONALITY ARCHETYPES")

for person in participants:
    print(f"{person} - {archetypes[person]}")




GROUPDNA REPORT — "Hostel Friends"
40 days • 324 messages • 6 members
Period : 01 April 2024 to 10 May 2024
Busiest day : 01/04/24 ( 13 messages)
Busiest hour : 18:00 - 19:00

MESSAGES PER PERSON
Rahul      ████████████████████ 70 (21.6%)
Priya      ████████████████████ 70 (21.6%)
Aman       ██████████████████ 64 (19.8%)
Neha       ████████████████ 57 (17.6%)
Karan      ██████████████ 51 (15.7%)
Vikas      ███ 12 (3.7%)

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
          00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul      . . . . . . . ▒ ▒ ▒ ▒ . . ▒ ▒ . . ░ █ . . . ▒ . 
Priya      . . . . . . . . ▒ ▒ █ ▒ █ . . . ▒ ░ . ▒ ▒ . . . 
Aman       . . . . . . . . . . ▒ ▒ ▒ ▒ . . . ░ ▒ ▒ ▒ . ▒ ▒ 
Karan      . . . . . . . . . . . . █ . ▒ ▒ . ░ █ . ▒ . . . 
Neha       . . . . . . . . . █ █ . . . . ▒ ▒ ░ . . . █ . . 
Vikas      . . . . . . . ▒ . . . . . . . . . . . . . . . ░ 

THIS GROUP'S FAVOURITE WORDS
good         ████████████████████ 46
yes          ████